In [270]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from pprint import pprint

llm = ChatOllama(model="llama3.2:3b")

#embedding_model = OllamaEmbeddings(model="nomic-embed-text")
embedding_model = OllamaEmbeddings(model="mxbai-embed-large")


events_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="events",
    embedding_function=embedding_model
)

sections_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="sections",
    embedding_function=embedding_model
)

In [281]:
query = "What are the technical specs of the Nintendo Switch 2 game console?"
#query = "Tell me the news of Nintendo Switch 2"
#query = "What was the market reception of the Nintendo Switch 2 game console?"
#query = "Tell me about Nintendo gaming consoles"
#query = "Compare the features of Nintendo Switch 2 to the original Nintendo Switch"
#query = "What are achievements of Prema Racing?"

raw_prompt = ChatPromptTemplate.from_template( 
    """
    Answer this query given all the knowledge you have:
    {query} 
    """
)

chain = raw_prompt | llm | StrOutputParser()
raw_output = chain.invoke({
    "query": query
})

print(f"Query: {query}\n")
print("Raw output without using RAG methods:\n")
print(raw_output)

print("\n\n\nRunning RAG pipeline...")


query_vector = embedding_model.embed_query(query)

top_events = events_vector_store.similarity_search_by_vector(
    query_vector, 
    k=5
)

event_index = {}
section_index = {}

#pprint(top_events)

formatted_events = ""

for event in top_events:
    #print(event)
    event_index[event.id] = event
    # don't use the question and reinsert the date
    event.page_content = event.page_content.split('?\n\n', 1)[1]  
    event.page_content = f"{event.metadata['day']} {event.metadata['month']}, {event.metadata['year']}: {event.page_content}"
    formatted_events += f"\n\n\nEvent {event.id}: {event.page_content}"
    top_sections = sections_vector_store.similarity_search_by_vector(
        query_vector,
        k=5,
        ids=event.metadata["sections"]
    )
    for section in top_sections:
        formatted_events += f"\n\t{event.id}.{section.id}: {section.metadata["title"]}"
        section_index[section.id]=section
    

select_events_prompt = ChatPromptTemplate.from_template( 
    """
    I was this query by the user: {query}

    These events and their sections could be relevant but some of them are not relevant:
        {formatted_events}

    Select only the events relevant to the user's query using a maximum of 5 events and 30 sections. Skip events not directly relevant to the user's query.
    
    Give the output in a JSON format strictly like this example: {{"e115": ["s899", "s655"], "e782": ["s311"]}}

    Only use event IDs in keys and nothing else.
    Only use section IDs for section elements and nothing else.
    
    Just output one object in the JSON and nothing else! 
    """
)

filled_select_events_prompt = select_events_prompt.format(
    query = query,
    formatted_events = formatted_events
)

# print(filled_select_events_prompt)

print(f"\nSelect events prompt length: {len(filled_select_events_prompt)}\n\n")


chain = select_events_prompt | llm | StrOutputParser()
json_str = chain.invoke({
    "query": query,
    "formatted_events": formatted_events
})

clean_json = json_str[json_str.find("{"):json_str.rfind("}") + 1]

print(f"JSON: {clean_json}")

import json

retrieved_content = ""

try:
    data = json.loads(clean_json)
    for ei, (event_id, sections) in enumerate(data.items(), 1):
        event = event_index.get(event_id)
        if event is None: 
            continue
            
        retrieved_content += f"{event.page_content} "
        for ej, section_id in enumerate(sections, 1):
            section_id = section_id.split(".", 1)[-1] # sometimes the format returned is e12313.s2343
            section = section_index.get(section_id)
            if section is None:
                continue
            retrieved_content += f"{section.metadata['summary']} "
        retrieved_content += "\n\n"
    
    print("\nRetrieved: \n")
    print(retrieved_content)

except Exception:
    data = None
    retrieved_content = "No articles found"

# Add today's date to the prompt so that LLM does not infer events in the future
from datetime import datetime
today = datetime.now().strftime("%Y-%m-%d")

final_prompt = ChatPromptTemplate.from_template( 
    """
    Todays date: {today}
    
    I was asked this query by the user: {query}

    Answer the user's query using only relevant parts of the below events that may not be connected to each other:
    {retrieved_content}

    Add factual information from your own knowledge-base to enhance the provided events but do not speculate.
    If no relevant events are listed, use your own knowledge-base to answer. Ignore events that are not connected.
    Just answer without mentioning news was provided to you but use the news.
    """
)

filled_final_prompt = final_prompt.format(
    query =  query,
    retrieved_content = retrieved_content,
    today = today
)

print(f"\nFinal prompt length: {len(filled_final_prompt)}\n\n")

chain = final_prompt | llm | StrOutputParser()

output = chain.invoke({
    "query": query,
    "retrieved_content": retrieved_content,
    "today": today
})

print("\n\nFinal output:\n")
print(output)

Query: What are the technical specs of the Nintendo Switch 2 game console?

Raw output without using RAG methods:

I don't have any information on a specific "Nintendo Switch 2" game console. The most recent iteration of the Nintendo Switch series is the Nintendo Switch OLED Model, which was released in October 2021.

However, I can provide you with the technical specs of the original Nintendo Switch and some rumored specifications for a potential Nintendo Switch 2:

Original Nintendo Switch (2017):

* Processor: NVIDIA Tegra X1+ SoC
* Graphics Processing Unit (GPU): Maxwell-based GPU
* RAM: 4GB GDDR5 RAM
* Storage: 32GB or 64GB internal storage (expandable via microSD cards)
* Display:
	+ Resolution: 720p (at 60fps) and 1080p (at 30fps)
	+ Aspect Ratio: 16:9
	+ Refresh Rate: Up to 120Hz (at 60fps)
* Battery Life: Approximately 2.5-6 hours

Rumored specifications for a potential Nintendo Switch 2:

* Processor: Custom AMD Zen 2 CPU, possibly with increased clock speeds and improved per